In [ ]:
# INSTALL DEPENDENCIES

!pip -q install pypdf                     # READ PDF
!pip -q install sentence-transformers     # SENTENCE --> NUMERICAL VECTORS
!pip -q install chromadb                  # STORAGE FOR VECTOR DATABASE
!pip -q install langchain                 # MAINTAIN CONNECTIVITY BETWEEN LLM AND DATA
!pip -q install langchain-text-splitters  # SPLIT DATA TO CHUNKS
!pip -q install transformers
!pip -q install accelerate
!pip -q install bitsandbytes
!pip -q install sentencepiece
!pip -q install deep-translator docling   # LANGUAGES PACKAGE
!pip install -q python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [ ]:
# UPLOAD THE NECESSARY PDF OF THE SCHEME
from google.colab import files

uploaded = files.upload()

Saving janani shishu scheme.pdf to janani shishu scheme.pdf


In [ ]:
import chromadb
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize local ChromaDB client
client = chromadb.Client()
collection_name = "multilingual_local_schemes"

try:
    client.delete_collection(name=collection_name)
    print(f"Previous Collection '{collection_name}' dropped.")
except Exception:
    print("Creating a fresh index...")

collection = client.create_collection(name=collection_name)

# 2. Configure Docling with Indian Language OCR support
pipeline_options = PdfPipelineOptions()
# Configured for English, Hindi, Kannada, Telugu, Tamil, and Marathi
pipeline_options.ocr_options.lang = ["en", "hi", "kn", "te", "ta", "mr"]

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# 3. Initialize the text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# 4. Extract text using Docling instead of PyPDF
pdf_file = list(uploaded.keys())[0]

print(f"Processing regional text layouts from '{pdf_file}'...")
conversion_result = converter.convert(pdf_file)
page_text = conversion_result.document.export_to_markdown()

# 5. Chunk and map to vector database
all_chunks = splitter.split_text(page_text)

stored_chunks = 0
for idx, chunk in enumerate(all_chunks):
    collection.add(
        ids=[f"chunk_{idx}"],
        documents=[chunk],
        metadatas=[{"document": pdf_file}]
    )
    stored_chunks += 1

print(f"\n--- SUCCESS ---")
print(f"Stored {stored_chunks} multilingual text blocks from '{pdf_file}' into ChromaDB.")

Creating a fresh index...
Processing regional text layouts from 'janani shishu scheme.pdf'...


[INFO] 2026-08-15 06:33:23,059 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 06:33:23,091 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-15 06:33:23,092 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-15 06:33:23,153 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 06:33:23,159 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-15 06:33:23,160 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-15 06:33:23,206 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-15 06:33:23,265 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

W0815 06:34:12.695000 617 torch/_inductor/utils.py:1731] [2/0_1] Not enough SMs to use max_autotune_gemm mode
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 35.9MiB/s]



--- SUCCESS ---
Stored 6 multilingual text blocks from 'janani shishu scheme.pdf' into ChromaDB.


In [ ]:
print(page_text[:2000])  # JUST PREVIEW TO VERIFY

Janani Shishu Suraksha Karyakram

DiagnosticsFree DietFree Transport For HospitalisationMedicineNewborn ChildPregnantWomen Check Eligibility Details Started in 2011, the scheme "Janani Shishu Suraksha Karyakram (JSSK)" by the Ministry of Health and Family Welfare, Government of India, aims to provide free and cashless delivery, treatment, diagnostics, drugs, diet, and transport services to pregnant women and sick newborns/infants at public health institutions. It also aims to eliminate out-of-pocket expenses related to delivery, treatment, and essential maternal and child healthcare services.

Benefits For Pregnant Women: Free and zero expense Delivery and Caesarean Section Free Drugs and Consumables Free Essential Diagnostics (Blood, Urine tests and Ultra-sonography etc) Free Diet during stay in the health institutions (up to 3 days for normal delivery &amp; 7 days for caesarean section) Free Provision of Blood Free Transport from Home to Health Institutions Free Transport between fac

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

chunk_data = []

# Split the entire document text into chunks
chunks_from_document = splitter.split_text(page_text)

# Assign a default page number (e.g., 1) to all chunks since page_text is a single string
for i, chunk in enumerate(chunks_from_document):
    chunk_data.append({
        "text": chunk,
        "page": 1 # All chunks come from the single 'page_text' string, so assign page 1
    })

print("Total Chunks:", len(chunk_data))

Total Chunks: 7


In [ ]:
print(chunk_data[3]) # VERIFY CHUNKS

{'text': 'Benefits For Pregnant Women: Free and zero expense Delivery and Caesarean Section Free Drugs and Consumables Free Essential Diagnostics (Blood, Urine tests and Ultra-sonography etc) Free Diet during stay in the health institutions (up to 3 days for normal delivery &amp; 7 days for caesarean section) Free Provision of Blood Free Transport from Home to Health Institutions Free Transport between', 'page': 1}


In [ ]:
#EMBEDDING MODEL SENTENCE -> EMBEDDING NUMERICAL VECTORS

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded


In [ ]:
texts = [chunk["text"] for chunk in chunk_data]
# ACTUAL EMBEDDING HAPPENS HERE
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (7, 384)


In [ ]:
import chromadb
# CREATE A DATA BASE COLLECTION TO STORE THE DATA
client = chromadb.Client()

collection_name = "government_schemes"

# Delete the collection if it already exists to ensure a clean state
try:
    client.delete_collection(name=collection_name)
    print(f"Previous Collection '{collection_name}' deleted successfully.")
except Exception as e:
    print(f"Could not delete collection '{collection_name}' or it did not exist: {e}")

# Create the collection
collection = client.create_collection(
    name=collection_name
)

print(f"Collection '{collection_name}' created successfully.")

Could not delete collection 'government_schemes' or it did not exist: Collection [government_schemes] does not exist
Collection 'government_schemes' created successfully.


In [ ]:
# ADD THE CHUNK_DATA TO THE COLLECTION MADE ABOVE

for i, chunk in enumerate(chunk_data):
    collection.add(
        ids=[str(i)],
        documents=[chunk["text"]],
        metadatas=[
            {
                "page": chunk["page"],
                "document": pdf_file
            }
        ]
    )

print("Stored", len(chunk_data), "chunks")

Stored 7 chunks


In [ ]:
question = input("Ask a question: ")  # ASK USER QUERRY

Ask a question: eligibility criteria 


In [ ]:
question_embedding = embedding_model.encode( # EMBEDDING THE QUERRY
    question
).tolist()

In [ ]:
results = collection.query(    # EXTRACT RESULTANT CHUNKS
    query_embeddings=[question_embedding],
    n_results=3
)

In [ ]:
unique_citations = set()
# CHECK FOR CITATIONS FROM THE UPLOADED PDF
for doc, meta in zip(
    results["documents"][0],
    results["metadatas"][0]
):

    print("=" * 80)
    document_name = "N/A"
    page_num = "N/A"

    if meta is not None:
        document_name = meta.get("document", "N/A")
        page_num = meta.get("page", "N/A")
        unique_citations.add((document_name, page_num))
        print("DOCUMENT:", document_name)
        print("PAGE:", page_num)
    else:
        print("METADATA: N/A (metadata was None)")
    print("=" * 80)

    print(doc)
    print("\n")

# Print citations at the end
if unique_citations:
    print("=" * 80)
    print("CITATIONS:")
    print("=" * 80)
    for doc_name, page_num in sorted(list(unique_citations)):
        print(f"Document: {doc_name}, Page: {page_num}")
else:
    print("No citations found.")

DOCUMENT: janani shishu scheme.pdf
PAGE: 1
Eligibility The applicant must be a pregnant woman. The applicant must be admitted to a government health facility.

Application Process Offline The pregnant woman is referred by the competent staff of the public health facility for JSSK.

Documents Required Aadhaar Number Address proof Domicile certificate Ration card Janani Suraksha Card Frequently Asked Q


DOCUMENT: janani shishu scheme.pdf
PAGE: 1
DiagnosticsFree DietFree Transport For HospitalisationMedicineNewborn ChildPregnantWomen Check Eligibility Details Started in 2011, the scheme "Janani Shishu Suraksha Karyakram (JSSK)" by the Ministry of Health and Family Welfare, Government of India, aims to provide free and cashless delivery, treatment, diagnostics, drugs, diet, and transport services to pregnant women and sick newborns/infants


DOCUMENT: janani shishu scheme.pdf
PAGE: 1
treatment Free Drugs and Consumables Free Diagnostics Free Provision of Blood Free Transport from Home to 

In [ ]:
for i, doc in enumerate(results["documents"][0]):
    print("=" * 80)
    print(f"RESULT {i+1}")
    print("=" * 80)
    print(doc)
    print("\n")

RESULT 1
Eligibility The applicant must be a pregnant woman. The applicant must be admitted to a government health facility.

Application Process Offline The pregnant woman is referred by the competent staff of the public health facility for JSSK.

Documents Required Aadhaar Number Address proof Domicile certificate Ration card Janani Suraksha Card Frequently Asked Q


RESULT 2
DiagnosticsFree DietFree Transport For HospitalisationMedicineNewborn ChildPregnantWomen Check Eligibility Details Started in 2011, the scheme "Janani Shishu Suraksha Karyakram (JSSK)" by the Ministry of Health and Family Welfare, Government of India, aims to provide free and cashless delivery, treatment, diagnostics, drugs, diet, and transport services to pregnant women and sick newborns/infants


RESULT 3
treatment Free Drugs and Consumables Free Diagnostics Free Provision of Blood Free Transport from Home to Health Institutions Free Transport between facilities in case of referral Drop Back from Institutions 

In [ ]:
# LOAD THE LLM AND TOKENIZER

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Phi-3 loaded successfully")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Phi-3 loaded successfully


In [ ]:
context = "\n\n".join(
    results["documents"][0]
)

print(context[:1000])

Eligibility The applicant must be a pregnant woman. The applicant must be admitted to a government health facility.

Application Process Offline The pregnant woman is referred by the competent staff of the public health facility for JSSK.

Documents Required Aadhaar Number Address proof Domicile certificate Ration card Janani Suraksha Card Frequently Asked Q

DiagnosticsFree DietFree Transport For HospitalisationMedicineNewborn ChildPregnantWomen Check Eligibility Details Started in 2011, the scheme "Janani Shishu Suraksha Karyakram (JSSK)" by the Ministry of Health and Family Welfare, Government of India, aims to provide free and cashless delivery, treatment, diagnostics, drugs, diet, and transport services to pregnant women and sick newborns/infants

treatment Free Drugs and Consumables Free Diagnostics Free Provision of Blood Free Transport from Home to Health Institutions Free Transport between facilities in case of referral Drop Back from Institutions to home Exemption from all ki

In [ ]:

# prompt = f"""
# You are an expert Government Scheme Assistant.

# Answer ONLY from the provided context.

# If the answer is not found in the context, reply exactly:

# Information not found in the uploaded document.

# Context:
# {context}

# Question:
# {question}

# Answer:
# """



# 1. Format the prompt using Phi-3's official chat structure
messages = [
    {
        "role": "system",
        "content": f"You are an expert Government Scheme Assistant. Answer ONLY from the provided context. If the answer is not found in the context, reply exactly: Information not found in the uploaded document.\n\nContext:\n{context}"
    },
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 2. Tokenize with attention mask
inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)

# 3. Generate clean output (FIXED: Removed duplicate attention_mask line)
outputs = model.generate(
    **inputs,                               # This automatically includes input_ids AND attention_mask!
    max_new_tokens=200,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# 4. Decode just the new tokens
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the assistant's final response line cleanly
answer = generated_text.split("assistant")[-1] if "assistant" in generated_text else generated_text[len(prompt):]

print(answer.strip())

uraksha Karyakram (JSSK) scheme are as follows:

1. The applicant must be a pregnant woman.

2. The applicant must be admitted to a government health facility.


The application process for the JSSK scheme is conducted offline. A pregnant woman is referred by the competent staff of the public health facility for the scheme.


The documents required for the JSSK scheme include:

- Aadhaar Number

- Address proof

- Domicile certificate

- Ration card

- Janani Suraksha Yojana (JSY) card


The JSSK scheme provides the following benefits:

- Free delivery

- Free treatment

- Free diagnostics

- Free drugs and consumables

- Free provision of blood


In [ ]:
# inputs = tokenizer(
#     prompt,
#     return_tensors="pt"
# ).to(model.device)

# outputs = model.generate(
#     **inputs,
#     max_new_tokens=200,
#     temperature=0.1,
#     do_sample=False
# )

# generated_text = tokenizer.decode(
#     outputs[0],
#     skip_special_tokens=True
# )

# answer = generated_text[len(prompt):]

# print(answer.strip())

In [ ]:
print("\nSOURCES:\n")

for meta in results["metadatas"][0]:
    if meta is not None:
        print(f'{meta.get("document", "N/A")} (Page {meta.get("page", "N/A")})')
    else:
        print("Source information N/A (metadata was None)")


SOURCES:

janani shishu scheme.pdf (Page 1)
janani shishu scheme.pdf (Page 1)
janani shishu scheme.pdf (Page 1)


In [ ]:
# The next cells were supposed to link this RAG Module to the frontend of the project using FAST API.

!pip install -q fastapi uvicorn pyngrok python-multipart
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import shutil

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    session_id: str
    question: str

class EligibilityRequest(BaseModel):
    session_id: str
    age: int
    annual_income: float
    state: str
    category: str
    occupation: str

CURRENT_DOCUMENT = pdf_file if 'pdf_file' in globals() else "document.pdf"

@app.get("/schemes")
def get_schemes():
    return {
        "schemes": [{
            "name": CURRENT_DOCUMENT,
            "locations": 1
        }]
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...), session_id: str = Form(...), permanent: bool = Form(...)):
    global CURRENT_DOCUMENT, collection
    try:
        file_path = file.filename
        CURRENT_DOCUMENT = file_path

        # Save file to Colab disk
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        # Convert using your existing Docling converter
        conversion_result = converter.convert(file_path)
        extracted_text = conversion_result.document.export_to_markdown()

        # Split text into chunks
        new_chunks = splitter.split_text(extracted_text)

        # Re-index into ChromaDB collection
        collection_name = "government_schemes"
        try:
            client.delete_collection(name=collection_name)
        except Exception:
            pass

        collection = client.create_collection(name=collection_name)
        for idx, chunk in enumerate(new_chunks):
            collection.add(
                ids=[str(idx)],
                documents=[chunk],
                metadatas=[{"page": 1, "document": file_path}]
            )

        return {"chunks": len(new_chunks), "filename": file.filename}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/chat")
def chat_with_agent(request: QueryRequest):
    user_query = request.question

    # Embed query and search ChromaDB
    q_embedding = embedding_model.encode(user_query).tolist()
    results = collection.query(query_embeddings=[q_embedding], n_results=3)

    retrieved_docs = results["documents"][0]
    retrieved_metas = results["metadatas"][0]
    context = "\n\n".join(retrieved_docs)

    # Format messages for Phi-3 chat template
    messages = [
        {
            "role": "system",
            "content": f"You are an expert Government Scheme Assistant. Answer ONLY from the provided context. If the answer is not found in the context, reply exactly: Information not found in the uploaded document.\n\nContext:\n{context}"
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    final_answer = generated_text[len(prompt):].strip()

    citations = []
    for meta in retrieved_metas:
        if meta:
            citations.append({
                "document": meta.get("document", CURRENT_DOCUMENT),
                "page": str(meta.get("page", "1"))
            })

    return {
        "answer": final_answer,
        "citations": list({c['document'] + c['page']: c for c in citations}.values()),
        "confidence": "High"
    }

@app.post("/eligibility")
def check_eligibility(req: EligibilityRequest):
    return {
        "answer": f"Based on your profile details (Age: {req.age}, Income: ₹{req.annual_income}, State: {req.state}), you meet the initial eligibility evaluation criteria.",
        "citations": [{"document": CURRENT_DOCUMENT, "page": "1"}],
        "confidence": "Medium"
    }

@app.delete("/uploads/{session_id}")
def clear_uploads(session_id: str):
    return {"status": "success"}

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_api, daemon=True).start()
print("FastAPI Backend is active and listening for frontend requests!")

FastAPI Backend is active and listening for frontend requests!


In [ ]:
# --- RUN THIS IN THE NEXT CELL TO GET YOUR PUBLIC URL ---
!pip install -q pyngrok
from pyngrok import ngrok

# 1. Set your actual ngrok authtoken here inside the quotes
ngrok.set_auth_token("3GoeNK8BzNZhwl9ytq5gR4nhe7r_tVRFUk8F6rLUDW2kYAuE")

# 2. Open a public tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"👉 LIVE BACKEND URL: {public_url}/chat")

INFO:     Started server process [617]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


👉 LIVE BACKEND URL: NgrokTunnel: "https://magical-vocalist-traitor.ngrok-free.dev" -> "http://localhost:8000"/chat
